In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report


In [2]:
df = pd.read_csv("/content/diabetic_data.csv")

print(df.shape)
print(df.head())

(101766, 50)
   encounter_id  patient_nbr             race  gender      age weight  \
0       2278392      8222157        Caucasian  Female   [0-10)      ?   
1        149190     55629189        Caucasian  Female  [10-20)      ?   
2         64410     86047875  AfricanAmerican  Female  [20-30)      ?   
3        500364     82442376        Caucasian    Male  [30-40)      ?   
4         16680     42519267        Caucasian    Male  [40-50)      ?   

   admission_type_id  discharge_disposition_id  admission_source_id  \
0                  6                        25                    1   
1                  1                         1                    7   
2                  1                         1                    7   
3                  1                         1                    7   
4                  1                         1                    7   

   time_in_hospital  ... citoglipton insulin  glyburide-metformin  \
0                 1  ...          No      No        

In [4]:
df = df.replace("?", np.nan)

print(df.isnull().sum())




encounter_id                    0
patient_nbr                     0
race                         2273
gender                          0
age                             0
weight                      98569
admission_type_id               0
discharge_disposition_id        0
admission_source_id             0
time_in_hospital                0
payer_code                  40256
medical_specialty           49949
num_lab_procedures              0
num_procedures                  0
num_medications                 0
number_outpatient               0
number_emergency                0
number_inpatient                0
diag_1                         21
diag_2                        358
diag_3                       1423
number_diagnoses                0
max_glu_serum               96420
A1Cresult                   84748
metformin                       0
repaglinide                     0
nateglinide                     0
chlorpropamide                  0
glimepiride                     0
acetohexamide 

In [5]:
df["target"] = (df["readmitted"] == "<30").astype(int)

print(df["target"].value_counts())



target
0    90409
1    11357
Name: count, dtype: int64


In [6]:
df = df.drop(columns=[
    "encounter_id",
    "patient_nbr",
    "weight",
    "payer_code",
    "medical_specialty",
    "readmitted"
])


In [7]:
num_cols = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

cat_cols = [
    "race",
    "gender",
    "age",
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
    "max_glu_serum",
    "A1Cresult",
    "metformin",
    "insulin",
    "change",
    "diabetesMed",
    "diag_1"
]


In [8]:
top_diag = df["diag_1"].value_counts().head(15).index
df["diag_1"] = df["diag_1"].where(df["diag_1"].isin(top_diag), "Other")

df["admission_type_id"] = df["admission_type_id"].astype(str)
df["discharge_disposition_id"] = df["discharge_disposition_id"].astype(str)
df["admission_source_id"] = df["admission_source_id"].astype(str)

df["race"] = df["race"].fillna("Unknown")
df["max_glu_serum"] = df["max_glu_serum"].fillna("Not measured")
df["A1Cresult"] = df["A1Cresult"].fillna("Not measured")

In [9]:
X = df[num_cols + cat_cols]
y = df["target"]

X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

print(X.shape)
print(y.shape)

(101766, 101)
(101766,)


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(81412, 101)
(20354, 101)


In [11]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)



In [13]:
model_l2 = LogisticRegression(
    penalty="l2",
    C=1.0,
    max_iter=1000,
    class_weight="balanced"
)
model_l2.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [14]:
model_no_l2 = LogisticRegression(
    penalty=None,
    max_iter=1000,
    class_weight="balanced"
)

model_no_l2.fit(X_train, y_train)



LogisticRegression(class_weight='balanced', max_iter=1000, penalty=None)

In [15]:
pred_l2 = model_l2.predict(X_test)
prob_l2 = model_l2.predict_proba(X_test)[:, 1]

pred_no_l2 = model_no_l2.predict(X_test)
prob_no_l2 = model_no_l2.predict_proba(X_test)[:, 1]



In [16]:
print("L2 Regularization")

print("ROC-AUC:", roc_auc_score(y_test, prob_l2))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, pred_l2))

print("\nClassification Report:")
print(classification_report(y_test, pred_l2, digits=3))


L2 Regularization
ROC-AUC: 0.6763738505744817

Confusion Matrix:
[[12311  5772]
 [  971  1300]]

Classification Report:
              precision    recall  f1-score   support

           0      0.927     0.681     0.785     18083
           1      0.184     0.572     0.278      2271

    accuracy                          0.669     20354
   macro avg      0.555     0.627     0.532     20354
weighted avg      0.844     0.669     0.728     20354



In [17]:
print("No Regularization")

print("ROC-AUC:", roc_auc_score(y_test, prob_no_l2))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, pred_no_l2))

print("\nClassification Report:")
print(classification_report(y_test, pred_no_l2, digits=3))

No Regularization
ROC-AUC: 0.6763264153089479

Confusion Matrix:
[[12312  5771]
 [  969  1302]]

Classification Report:
              precision    recall  f1-score   support

           0      0.927     0.681     0.785     18083
           1      0.184     0.573     0.279      2271

    accuracy                          0.669     20354
   macro avg      0.556     0.627     0.532     20354
weighted avg      0.844     0.669     0.729     20354



In [18]:
comparison = pd.DataFrame({
    "Model": [
        "L2 Regularization",
        "No Regularization"
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, prob_l2),
        roc_auc_score(y_test, prob_no_l2)
    ]
})

print(comparison)

               Model   ROC-AUC
0  L2 Regularization  0.676374
1  No Regularization  0.676326
